In [45]:
import numpy as np
from typing import Tuple
from sklearn.tree import DecisionTreeRegressor
import matplotlib.pyplot as plt


def step_function(x):
    y_true = np.piecewise(
        x,
        [
            x < 0.35,
            (x >= 0.35) & (x < 0.45),
            (x >= 0.45) & (x < 0.55),
            (x >= 0.55) & (x < 0.65),
            x >= 0.65,
        ],
        [0.0, 0.7, 1.4, 0.7, 0.0],
    )
    return y_true

def generate_response(  x_train: np.ndarray, 
                        loc = 0,
                        var=0.25,
                        seed: int = None):

    rng = np.random.default_rng(seed)
    n_x = x_train.shape[0]

    noise = rng.normal(loc=loc, scale=np.sqrt(var), size=n_x)
    y_true = step_function(x_train)
    y_train = y_true + noise

    return y_train

def create_bootstrap_indices_and_Nbi(
    n: int, B: int, seed: int = None, boot_weights: np.ndarray = None
):
    if boot_weights is None:
        rng = np.random.default_rng(seed)
        boot_indices = rng.choice(np.arange(n), size=(B, n), replace=True)
        boot_counts = np.apply_along_axis(
            lambda x: np.bincount(x, minlength=n), axis=1, arr=boot_indices
        )
        return boot_indices, boot_counts

    else:
        rng = np.random.default_rng(seed)
        boot_indices = rng.choice(np.arange(n), size=(B, n), p=boot_weights, replace=True)
        boot_counts = np.apply_along_axis(
            lambda x: np.bincount(x, minlength=n), axis=1, arr=boot_indices
        )
        return boot_indices, boot_counts

def fit_tree(x_train, y_train, boot_indices, decision_tree_args):
    B, _ = boot_indices.shape
    trees = []
    for b in range(B):
        tree = DecisionTreeRegressor(**decision_tree_args)
        tree.fit(x_train[boot_indices[b]], y_train[boot_indices[b]])
        trees.append(tree)
    return trees

In [46]:
# Params
seed = 42
n = 500
B = 100

In [47]:
rng = np.random.default_rng(seed)
decision_tree_args = {"max_leaf_nodes": 5}

# Generate data from a uniform distribution with limits [0, 1]
x_train = rng.uniform(0, 1, n)
x_grid = np.linspace(0, 1, 1000).reshape(-1, 1)

# training data
y_train = generate_response(x_train=x_train, seed=seed)

# weight distribution
weights = np.zeros(n)
weights += 1 / int(n/2)
index_drop = rng.choice(range(n), size=int(n/2), replace=False)
weights[index_drop] = 0.0

# bootstrap samples and count vector
weighted_boot_indices, weighted_boot_counts = create_bootstrap_indices_and_Nbi(n=n, B=B, seed=seed, boot_weights=weights)
boot_indices, boot_counts = create_bootstrap_indices_and_Nbi(n=n, B=B, seed=seed)

# fit regression tree
weighted_trees = fit_tree(x_train.reshape(-1, 1), y_train, weighted_boot_indices, decision_tree_args)
trees = fit_tree(x_train.reshape(-1, 1), y_train, boot_indices, decision_tree_args)

In [50]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# Erste Grafik: Weighted Bootstraps
axes[1].set_title("(B) Trees fitted with Weighted boot-samples")
for tree in weighted_trees:
    y_pred = tree.predict(x_grid)
    axes[1].plot(x_grid, y_pred, color='gray', alpha=0.3)

axes[1].plot(x_grid, step_function(x_grid), color='red', lw=2, label='true step function')
axes[1].set_xlabel("x")

axes[1].legend()

# Zweite Grafik: Unweighted Bootstraps
axes[0].set_title("(A) Trees fitted with UNweighted boot-samples")
for tree in trees:
    y_pred = tree.predict(x_grid)
    axes[0].plot(x_grid, y_pred, color='gray', alpha=0.3)

axes[0].plot(x_grid, step_function(x_grid), color='red', lw=2, label='true step function')
axes[0].set_xlabel("x")
axes[0].set_ylabel("prediction")
axes[0].legend()

plt.suptitle("Estimated Step Functions of 100 Trees")
plt.tight_layout()
plt.savefig("example1.png", bbox_inches='tight')
plt.close()